In [ ]:
#BME 450 Final Project

#Input city, weather, and gender of customer
#NN will predict the product name that is most likely to be purchased

In [27]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [28]:
#upload dataset to colab
from google.colab import files
upload = files.upload()

Saving coffee_shop_sales.csv to coffee_shop_sales (2).csv


In [29]:
#import dataset
data = pd.read_csv("coffee_shop_sales.csv")

In [30]:
#display the column labels for the dataset
print(data.columns) #display the index for the dataset columns

Index(['transaction_id', 'timestamp', 'store_id', 'city', 'country',
       'store_type', 'product_category', 'product_name', 'unit_price',
       'quantity', 'discount_applied', 'payment_method', 'customer_id',
       'customer_age_group', 'customer_gender', 'loyalty_member',
       'weather_condition', 'temperature_c', 'holiday_name', 'total_amount'],
      dtype='object')


In [31]:
#creates groups in the dataset based on the desired input conditions and desired output
condense = data.groupby(['city', 'weather_condition', 'customer_gender', 'product_name'], as_index=False)['total_amount'].sum()

In [32]:
#function to determine the top purchased item category based on the input conditions
def find_top_purchased(x):
  return x.loc[x['total_amount'].idxmax()]

#finds top purchased items using defined function and inputs of city, weather, and gender
top_purchased = condense.groupby(['city', 'weather_condition', 'customer_gender']).apply(find_top_purchased).reset_index(drop=True)

/tmp/ipykernel_18470/1794530268.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_purchased = condense.groupby(['city', 'weather_condition', 'customer_gender']).apply(find_top_purchased).reset_index(drop=True)


In [33]:
#input definition and target definition
conditions = top_purchased[['city', 'customer_gender', 'weather_condition']]
target = top_purchased['product_name']

In [34]:
#convert column values into integers for easier processing
feature_encoders = {}
for col in ['city', 'weather_condition', 'customer_gender']:
  le = LabelEncoder()
  conditions[col] = le.fit_transform(conditions[col])
  feature_encoders[col] = le

target_encoder = LabelEncoder()
target =target_encoder.fit_transform(target)
num_classes = len(target_encoder.classes_)

/tmp/ipykernel_18470/446655162.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  conditions[col] = le.fit_transform(conditions[col])
/tmp/ipykernel_18470/446655162.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  conditions[col] = le.fit_transform(conditions[col])
/tmp/ipykernel_18470/446655162.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyd

In [1]:
#create tensors for the inputs and outputs
conditions_tensor = torch.tensor(conditions.values, dtype=torch.float32) #shape is determined by the number of samples and features
target_tensor = torch.tensor(target, dtype=torch.long)

NameError: name 'torch' is not defined

In [40]:
#definition of the neural network
class PurchaseClassifier(nn.Module):
  def __init__(self, input_dim, num_classes):
    super(PurchaseClassifier, self).__init__()
    self.model = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, num_classes) #output logits for each product
    )

  def forward(self, x):
    return self.model(x)

input_dim = conditions_tensor.shape[1]
model = PurchaseClassifier(input_dim, num_classes)

In [41]:
#funtions to calculate loss and optimize the weights of the NN to minimize loss
find_loss = nn.CrossEntropyLoss()
optimize = torch.optim.Adam(model.parameters(), lr=0.01)

In [42]:
#training the model
epochs = 100

#loop through epochs for training while calculating the loss
for epoch in range(epochs):
  model.train()
  optimize.zero_grad()

  logits = model(conditions_tensor)
  loss = find_loss(logits, target_tensor)

  #backwards propogation
  loss.backward()
  optimize.step()

  #prints epoch and loss every 10 epochs
  if (epoch + 1) % 10 == 0:
    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 2.5945
Epoch 20, Loss: 2.2764
Epoch 30, Loss: 2.0127
Epoch 40, Loss: 1.7539
Epoch 50, Loss: 1.5044
Epoch 60, Loss: 1.2873
Epoch 70, Loss: 1.1028
Epoch 80, Loss: 0.9500
Epoch 90, Loss: 0.8252
Epoch 100, Loss: 0.7322


In [43]:
#function to determine predicted purchased item based on user input of conditions
def predict_top_purchased(model, city_input, weather_input, gender_input, feature_encoders, target_encoder):

  #Encoded user input
  city_encoded = feature_encoders['city'].transform([city_input])[0]
  weather_encoded = feature_encoders['weather_condition'].transform([weather_input])[0]
  gender_encoded = feature_encoders['customer_gender'].transform([gender_input])[0]

  #Input tensor
  input_tensor = torch.tensor([[city_encoded, weather_encoded, gender_encoded]], dtype=torch.float32)

  model.eval()
  with torch.no_grad():
    logits = model(input_tensor)
    predicted_class = torch.argmax(logits, dim=1).item()
    predicted_purchase = target_encoder.inverse_transform([predicted_class])[0]

  return predicted_purchase

In [53]:
#User inputs for conditions
input_city = input("Enter city name: ")
input_weather = input("Enter weather condition in city: ")
input_gender = input("Enter gender of the customer: ")

#Run function with submitted inputs
purchased_item = predict_top_purchased(model, input_city, input_weather, input_gender, feature_encoders, target_encoder)

#print the predicted top item purchased based on these conditions
print(f"The customer is most likely to purchase: {purchased_item}")

Enter city name: Vancouver
Enter weather condition in city: Sunny
Enter gender of the customer: Male
The customer is most likely to purchase: Small Latte
